In [17]:
#importing the libraries
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import accuracy_score
import re

In [18]:
train_df = pd.read_csv("./phm_train.csv")
test_df = pd.read_csv("./phm_test.csv")

texts_train = train_df["tweet"].values
labels_train = train_df["label"].values

texts_test = test_df["tweet"].values
labels_test = test_df["label"].values

In [19]:
#data preprocess
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    tokens = text.split()
    return tokens

texts_train = [preprocess_text(t) for t in texts_train]
texts_test = [preprocess_text(t) for t in texts_test]

In [7]:
def build_vocab(tokenized_texts, max_vocab=15000):
    counter = Counter()
    for tokens in tokenized_texts:
        counter.update(tokens)

    vocab = {"<PAD>": 0, "<UNK>": 1}
    most_common = counter.most_common(max_vocab - 2)

    for i, (word, _) in enumerate(most_common):
        vocab[word] = i + 2

    return vocab

vocab = build_vocab(texts_train)

In [8]:
def tokens_to_sequence(tokens, vocab):
    return [vocab.get(word, vocab["<UNK>"]) for word in tokens]

MAX_LEN = 60

def pad_sequence(seq, max_len):
    if len(seq) < max_len:
        return seq + [0] * (max_len - len(seq))
    return seq[:max_len]

X_train = [pad_sequence(tokens_to_sequence(t, vocab), MAX_LEN) for t in texts_train]
X_test = [pad_sequence(tokens_to_sequence(t, vocab), MAX_LEN) for t in texts_test]

In [9]:
class TweetDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [10]:
train_loader = DataLoader(TweetDataset(X_train, labels_train),
                          batch_size=64, shuffle=True)

test_loader = DataLoader(TweetDataset(X_test, labels_test),
                         batch_size=64)


In [14]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64):
        super(LSTMModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(embed_dim,
                            hidden_dim,
                            batch_first=True,
                           num_layers = 1)

        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        out = self.dropout(h_n[-1])
        out = self.fc(out)
        return out

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMModel(len(vocab)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

EPOCHS = 3

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
        optimizer.step()

        total_loss += loss.item()

        predictions = torch.sigmoid(outputs) > 0.5
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

    print(f"Epoch {epoch+1}")
    print("Train Loss:", total_loss / len(train_loader))
    print("Train Accuracy:", correct / total)
    print("-" * 30)



Epoch 1
Train Loss: 0.6082537715222426
Train Accuracy: 0.7087378640776699
------------------------------
Epoch 2
Train Loss: 0.6039335655558641
Train Accuracy: 0.7097387648883996
------------------------------
Epoch 3
Train Loss: 0.6033946528176594
Train Accuracy: 0.7097387648883996
------------------------------


In [16]:
model.eval()
total_loss = 0
correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).unsqueeze(1)

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        total_loss += loss.item()

        predictions = torch.sigmoid(outputs) > 0.5

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

print("Test Loss:", total_loss / len(test_loader))
print("Test Accuracy:", correct / total)


Test Loss: 0.5985341122690236
Test Accuracy: 0.709696787751426
